# Какие технологии просят чаще всего

Первый вопрос к данным: какой стек встречается в вакансиях чаще всего и
меняется ли он от месяца к месяцу.

Стек в hh.ru не выделен отдельным полем — его приходится разбирать из
текста требований. Как это сделано и какие у этого способа ограничения —
в [README](../README.md). Здесь только сами числа.

In [ ]:
import sys
from pathlib import Path


def find_root() -> Path:
    """Корень репозитория: ищем pyproject.toml вверх по дереву."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from vacancy_dashboard import analysis, clean, collect  # noqa: E402

RAW = ROOT / "data" / "raw" / "vacancies.json"
FIXTURE = ROOT / "tests" / "fixtures" / "vacancies_sample.json"

source = RAW if RAW.exists() else FIXTURE
print(f"Источник данных: {source}")
if source == FIXTURE:
    print(
        "ВНИМАНИЕ: это синтетическая фикстура, а не рыночные данные. "
        "Числа отсюда нельзя публиковать."
    )

frame = clean.drop_archived(clean.build_frame(collect.load(source)))
print(f"Вакансий в выборке: {len(frame)}")
frame.head()

## Топ технологий

Считаются вакансии, а не упоминания: одна вакансия, где Python назван трижды, попадает в счётчик один раз.

In [ ]:
result, figure = analysis.question_stack(frame)
print(result.text)
result.table

In [ ]:
figure

## Как менялось по месяцам

Доля вакансий с каждой технологией — в процентах от числа вакансий, опубликованных за месяц.

In [ ]:
technologies = list(result.table["Технология"][:5])
months = analysis.stack_by_month(frame, technologies)
months.round(1)

In [ ]:
analysis.monthly_trend(frame, technologies)